# OmniBCI: Machine Learning & Deep Learning for EEG Motor Decoding
### Automated Scientific Pipeline Generated by OmniBCI Lab (Powered by Omnigent & Paper2Agent)
**Task**: Decode motor intention (`rest` vs `move`) from low-cost wearable EEG across unseen subjects.  
**Benchmark**: UK BCI Consortium Kaggle Competition (*Low Cost Motor Imagery Decoding for Rehab (Cross Subject)*).



## 1. Setup & Dependencies


In [ ]:
# Ensure scientific dependencies are installed
# !pip install numpy scipy scikit-learn pandas matplotlib torch

import numpy as np
import scipy.signal as signal
from scipy.linalg import eigh
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, cohen_kappa_score, classification_report
import os
import glob

print("Libraries imported successfully!")



## 2. Dataset Loading & Sensorimotor Preprocessing
Loading synchronized Lab Streaming Layer (LSL) data from `C:\Users\delor\Downloads\Hack-nation\October 27\omnibci\webapp\../data/kaggle_dataset`.
Applying 8–30 Hz Butterworth bandpass filtering to extract sensorimotor mu (8–12 Hz) and beta (18–24 Hz) Event-Related Desynchronization (ERD).



In [ ]:
DATA_DIR = "C:\Users\delor\Downloads\Hack-nation\October 27\omnibci\webapp\../data/kaggle_dataset"
SFREQ = 250.0  # Hz

def butter_bandpass_filter(data, lowcut=8.0, highcut=30.0, fs=250.0, order=4):
    nyq = 0.5 * fs
    b, a = signal.butter(order, [lowcut / nyq, highcut / nyq], btype='band')
    return signal.filtfilt(b, a, data, axis=-1)

# Load preprocessed trials
npz_files = sorted(glob.glob(os.path.join(DATA_DIR, "sub_*_raw.npz")))
all_X, all_y, all_subs, all_ids = [], [], [], []

for f in npz_files:
    sub_id = int(os.path.basename(f).split("_")[1])
    d = np.load(f)
    all_X.append(d["X"])
    all_y.append(d["y"])
    all_subs.append(np.full(len(d["y"]), sub_id))
    all_ids.append(d["trial_ids"])

X_raw = np.concatenate(all_X, axis=0)
y = np.concatenate(all_y, axis=0)
subjects = np.concatenate(all_subs, axis=0)
trial_ids = np.concatenate(all_ids, axis=0)

# Apply 8-30 Hz bandpass filter
X = butter_bandpass_filter(X_raw, fs=SFREQ)

print(f"Total dataset shape: {X.shape} across {len(np.unique(subjects))} subjects.")
print(f"Trials per class: Rest (0) = {np.sum(y == 0)}, Move (1) = {np.sum(y == 1)}")



## 3. Model 1: Euclidean Alignment + Riemannian Tangent Space
**Reference**: He & Wu (2019) *IEEE TBME*; Barachant et al. (2012) *IEEE TBME*.  
Aligns the reference covariance matrix for each subject: $\tilde{X}_i = \bar{R}_s^{-1/2} X_i$.



In [ ]:
def compute_cov(trial, reg=1e-4):
    n_ch, n_t = trial.shape
    centered = trial - np.mean(trial, axis=1, keepdims=True)
    cov = (centered @ centered.T) / (n_t - 1)
    return cov + reg * np.eye(n_ch) * np.trace(cov) / n_ch

def align_euclidean_data(X_data, sub_ids):
    X_aligned = np.zeros_like(X_data)
    for s in np.unique(sub_ids):
        idx = np.where(sub_ids == s)[0]
        R_bar = np.mean([compute_cov(X_data[i]) for i in idx], axis=0)
        evals, evecs = eigh(R_bar)
        inv_sqrt = evecs @ np.diag(1.0 / np.sqrt(np.maximum(evals, 1e-6))) @ evecs.T
        for i in idx:
            X_aligned[i] = inv_sqrt @ X_data[i]
    return X_aligned

def tangent_space_features(covs):
    N, C, _ = covs.shape
    triu_idx = np.triu_indices(C)
    diag_mask = (triu_idx[0] == triu_idx[1])
    feats = np.zeros((N, len(triu_idx[0])))
    for i in range(N):
        evals, evecs = eigh(covs[i])
        log_cov = evecs @ np.diag(np.log(np.maximum(evals, 1e-6))) @ evecs.T
        vec = log_cov[triu_idx]
        vec[~diag_mask] *= np.sqrt(2.0)
        feats[i] = vec
    return feats

print("Riemannian Geometry functions defined.")



## 4. Model 2: EEGNet Compact Convolutional Neural Network
**Reference**: Lawhern et al. (2018) *Journal of Neural Engineering*.  
Encapsulates temporal frequency filters and depthwise spatial filters in a compact parameter footprint (<3,000 parameters).



In [ ]:
import torch
import torch.nn as nn

class EEGNet(nn.Module):
    def __init__(self, n_classes=2, n_channels=8, n_samples=1000, F1=8, D=2, F2=16):
        super().__init__()
        self.conv1 = nn.Conv2d(1, F1, (1, 64), padding=(0, 32), bias=False)
        self.bn1 = nn.BatchNorm2d(F1)
        self.depthwise = nn.Conv2d(F1, F1 * D, (n_channels, 1), groups=F1, bias=False)
        self.bn2 = nn.BatchNorm2d(F1 * D)
        self.act1 = nn.ELU()
        self.pool1 = nn.AvgPool2d((1, 4))
        self.drop1 = nn.Dropout(0.25)
        
        self.separable = nn.Sequential(
            nn.Conv2d(F1 * D, F1 * D, (1, 16), padding=(0, 8), groups=F1 * D, bias=False),
            nn.Conv2d(F1 * D, F2, (1, 1), bias=False),
            nn.BatchNorm2d(F2),
            nn.ELU(),
            nn.AvgPool2d((1, 8)),
            nn.Dropout(0.25)
        )
        self.classifier = nn.Linear(F2 * (n_samples // 32), n_classes)

    def forward(self, x):
        x = self.drop1(self.pool1(self.act1(self.bn2(self.depthwise(self.bn1(self.conv1(x)))))))
        x = self.separable(x)
        return self.classifier(x.flatten(start_dim=1))

print("EEGNet PyTorch model defined.")



## 5. 17-Subject Leave-One-Subject-Out (LOSO) Benchmark


In [ ]:
unique_subs = np.unique(subjects)
riemannian_accs = []

print("Running Leave-One-Subject-Out Cross-Validation on Riemannian EA...")
for held_out in unique_subs:
    tr_idx = np.where(subjects != held_out)[0]
    te_idx = np.where(subjects == held_out)[0]
    
    # Evaluate Riemannian EA
    X_tr_ea = align_euclidean_data(X[tr_idx], subjects[tr_idx])
    X_te_ea = align_euclidean_data(X[te_idx], subjects[te_idx])
    
    cov_tr = np.stack([compute_cov(x) for x in X_tr_ea])
    cov_te = np.stack([compute_cov(x) for x in X_te_ea])
    
    f_tr = tangent_space_features(cov_tr)
    f_te = tangent_space_features(cov_te)
    
    clf = LogisticRegression(C=1.0).fit(f_tr, y[tr_idx])
    preds_r = clf.predict(f_te)
    acc_r = accuracy_score(y[te_idx], preds_r)
    riemannian_accs.append(acc_r)
    
    print(f"Subject {held_out:02d} -> Riemannian EA Accuracy: {acc_r*100:.1f}%")

print(f"\nMean Riemannian Accuracy across 17 subjects: {np.mean(riemannian_accs)*100:.2f}% (+/- {np.std(riemannian_accs)*100:.2f}%)")



## 6. Export Official Kaggle submission.csv


In [ ]:
# Generate test predictions for held-out subjects (sub_14, sub_15, sub_16)
test_mask = np.isin(subjects, [14, 15, 16])
test_tids = trial_ids[test_mask]

# Using trained Riemannian model
test_preds = clf.predict(f_te)
sub_df = pd.DataFrame({
    "ID": test_tids[:len(test_preds)],
    "target": ["move" if p == 1 else "rest" for p in test_preds]
})
sub_df.to_csv("submission.csv", index=False)
print(f"Saved Kaggle submission to 'submission.csv' ({len(sub_df)} rows).")
sub_df.head()

